# 15 — Benamou–Brenier and Otto

Watch the Wasserstein geodesic as a *flow* whose kinetic energy is exactly the squared distance $W_2^2$, and read what that says about the shape of the space of distributions.

**Prerequisites:** `03/05_wasserstein_distance`, `03/07_mccann_geodesic`, `03/12_bures_wasserstein`, `03/14_gaussian_geodesic`.

**What you'll be able to do**
- Write the continuity equation from a count of what crosses an interval's ends, and check it on the Gaussian geodesic.
- Compute the kinetic energy of a flow, and see that the geodesic flow between two Gaussians costs exactly $W_2^2$.
- State the theorem of Benamou and Brenier, explain with particles why no flow does better, and test it against rival flows.
- Read Otto's geometric picture, test the non-negative curvature inequality on Gaussians, and place the quantum counterpart on the course map.

You have seen the best path between two Gaussians. Now ask how fast the mass moves along it, and why no motion is cheaper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.transport.gaussian import (
    bures_wasserstein_distance,
    gaussian_geodesic,
    gaussian_ot_map,
)

rng = np.random.default_rng(0)   # seed for the 2,000 random covariance triples of section 4
viz.use_course_style()

## Where we are

`03/14` established that between two Gaussians the optimal map is affine and the geodesic stays inside the Gaussian family: snapshots $\mu_t$, each a Gaussian, run at constant speed. We keep its one-dimensional example $\mathcal N(0, 1) \to \mathcal N(4, 0.5^2)$, with map $T(x) = m_1 + (\sigma_1/\sigma_0)(x - m_0)$.

A list of snapshots does not tell you how the mass *moves*: how fast a grain travels, or why this path is the shortest. Those are the questions here, and they lead to a second description of $W_2$, as the least energy of a motion.

## 1. Mass in motion: the continuity equation

Picture a density $\rho(t, x)$ on the line, the mass per unit length, and a **velocity field** $v(t, x)$, the speed of the mass at $x$ at time $t$. Write $\partial_t$, $\partial_x$ for rates of change in time and space. The mass in a small interval $[a, b]$ changes only because mass crosses the two ends, at the rate density times velocity, $\rho v$, the **flux**:

$$ \frac{\mathrm d}{\mathrm dt}\int_a^b \rho\,\mathrm dx = \rho v\,\big|_{x=a} - \rho v\,\big|_{x=b} = -\int_a^b \partial_x(\rho v)\,\mathrm dx . $$

This holds for every interval, so the integrands agree: $\partial_t \rho + \partial_x(\rho v) = 0$, the **continuity equation**: mass is neither created nor destroyed. In several dimensions $\partial_x$ becomes the divergence $\nabla\!\cdot$. Below, $\rho_t$ is shorthand for $x \mapsto \rho(t, x)$, and a pair $(\rho_t, v_t)$ obeying the equation is a *flow*.

**By hand: a translation.** Let a fixed profile $f$ slide at speed $c$: $\rho(t, x) = f(x - ct)$ and $v \equiv c$. Then $\partial_t\rho = -c\,f'(x - ct)$ and $\partial_x(\rho v) = c\,f'(x - ct)$, and the two cancel.

### The geodesic as a flow

Each grain on the geodesic of `03/14` travels a straight track from $x_0$ to $T(x_0)$ at constant speed, so its velocity is the constant $T(x_0) - x_0$. Here $T(x) = 4 + 0.5\,x$, hence $4 - 0.5\,x_0$. Write $\mu_t = \mathcal N(m_t, \sigma_t^2)$ with $m_t = 4t$ and $\sigma_t = 1 - 0.5\,t$, and $\rho_t$ for its density; a dot means the derivative in $t$, so $\dot m_t = 4$ and $\dot\sigma_t = -0.5$. At time $t$ the grain that started at $x_0$ sits at $x = m_t + \sigma_t x_0$. Solving for $x_0$ gives the velocity at the place $x$:

$$ v(t, x) = 4 - 0.5\,\frac{x - m_t}{\sigma_t} = \dot m_t + \frac{\dot\sigma_t}{\sigma_t}\,(x - m_t). $$

The next cell checks on a fine grid that $\rho_t$ and this field obey the continuity equation (derivatives are differences between neighbouring grid values), tries a field that only translates the mass, and draws the picture.

In [ ]:
m0, s0, m1, s1 = 0.0, 1.0, 4.0, 0.5            # N(0, 1) -> N(4, 0.5^2), the example of 03/14

slope = float(gaussian_ot_map(np.array([[s0**2]]), np.array([[s1**2]]))[0, 0])
print(f"slope of the optimal map from the library: {slope:.4f}  (= s1/s0 = {s1 / s0:.4f})")
mt_lib, ct_lib = gaussian_geodesic(np.array([m0]), np.array([[s0**2]]), np.array([m1]), np.array([[s1**2]]), 0.5)
print(f"library midpoint: mean {mt_lib[0]:.4f}, standard deviation {np.sqrt(ct_lib[0, 0]):.4f}")


def mean(t):
    return m0 + t * (m1 - m0)


def std(t):
    return s0 + t * (s1 - s0)


def rho(t, x):
    return np.exp(-0.5 * ((x - mean(t)) / std(t)) ** 2) / (std(t) * np.sqrt(2 * np.pi))


def velocity(t, x):
    return (m1 - m0) + (s1 - s0) / std(t) * (x - mean(t))


# Continuity residual  d_t rho + d_x (rho v)  at t = 0.4, by centred differences on a fine grid.
x = np.linspace(-8, 12, 8001)
dx = x[1] - x[0]
t0, h = 0.4, 1e-4
d_rho_dt = (rho(t0 + h, x) - rho(t0 - h, x)) / (2 * h)
residual = d_rho_dt + np.gradient(rho(t0, x) * velocity(t0, x), dx)
residual_translation = d_rho_dt + np.gradient(rho(t0, x) * (m1 - m0), dx)
print(f"size of the time derivative, max |d_t rho|        : {np.abs(d_rho_dt).max():.2f}")
print(f"residual with the geodesic velocity               : {np.abs(residual).max():.1e}")
print(f"residual with a velocity that only translates     : {np.abs(residual_translation).max():.2f}")

# Figure 1: density in the plane (time, position) with the straight tracks, and the velocity field.
t_axis = np.linspace(0, 1, 300)
x_axis = np.linspace(-3, 7, 400)
density = np.array([rho(t, x_axis) for t in t_axis]).T          # shape (400, 300): position by time
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"wspace": 0.25})
ax = axes[0]
mesh = ax.pcolormesh(t_axis, x_axis, density, cmap=viz.CMAP_PLAN, shading="auto")
fig.colorbar(mesh, ax=ax, pad=0.02, label="density rho(t, x)")
for z in np.linspace(-2, 2, 9):                                  # one grain per standardised position z
    start = m0 + s0 * z
    ax.plot(t_axis, (1 - t_axis) * start + t_axis * (m1 + (s1 / s0) * (start - m0)), color=COLORS["text"], lw=1.1)
ax.set_xlabel("time t")
ax.set_ylabel("position x")
ax.set_title("Density and the straight tracks of nine grains", pad=12)
ax = axes[1]
for t, color in [(0.0, viz.SOURCE_COLOR), (0.5, viz.FLOW_COLOR), (1.0, viz.TARGET_COLOR)]:
    inside = np.abs(x_axis - mean(t)) <= 2.5 * std(t)           # draw the field where the mass actually is
    ax.plot(x_axis[inside], velocity(t, x_axis[inside]), color=color, lw=2.6, label=f"t = {t:.1f}")
ax.set_xlabel("position x")
ax.set_ylabel("velocity v(t, x)")
ax.set_title("The velocity field, where the mass is", pad=12)
ax.legend(loc="upper right")
plt.show()

**Read the figure.** The continuity residual is $5.6\times10^{-6}$, against a time derivative of size $1.5$: $\rho_t$ and $v$ obey the equation. The translating field leaves $0.31$, a fifth of it: the bump shrinks as well as moves, so grains at its right edge must be slower than those at its left edge. On the left, time runs horizontally and position vertically. The green band is the density; it drifts up and narrows. The nine dark lines are the tracks of grains at standardised positions $z$ from $-2$ to $2$: all straight, as in `03/07`, and drawing closer as the mass is squeezed. On the right, each velocity line is drawn only where the mass is. At $t = 0$ it is $4 - 0.5\,x$: a grain at $x = -2$ runs at speed $5$, one at $x = 2$ at speed $3$, and each keeps its speed along its track.

## 2. The kinetic energy of a flow

Each grain of mass contributes the square of its speed, so a flow $(\rho_t, v_t)$ has the kinetic energy

$$ \mathcal E(\rho, v) = \int_0^1 \!\!\int |v(t, x)|^2\,\rho(t, x)\,\mathrm dx\,\mathrm dt . $$

The mass is one, so this is an average; the physicists' factor $\tfrac12$ is dropped.

**By hand: a translation.** Carried at the constant velocity $d$ for unit time, every grain has speed $d$: the energy is $d^2$, which is $W_2^2$ by `03/12`.

**By hand: the Gaussian flow.** A grain $x_0 = z \sim \mathcal N(0, 1)$ has the constant velocity $4 - 0.5\,z$, so the energy is the average of its square:

$$ \mathcal E = \mathbb E\big[(4 - 0.5\,z)^2\big] = 16 - 4\,\mathbb E[z] + 0.25\,\mathbb E[z^2] = 16 + 0.25 = 16.25 . $$

For any curve of Gaussians whose grains keep their standardised position $z$, the velocity is $\dot m_t + \dot\sigma_t z$ and the speed averages to

$$ \int |v|^2\rho\,\mathrm dx = \dot m_t^2 + \dot\sigma_t^2 . $$

For the geodesic this is $16.25$ at every instant, and by `03/12` $W_2^2 = (m_1 - m_0)^2 + (\sigma_1 - \sigma_0)^2 = 16.25$ too.

**Which flow?** On the line, when $\rho_t > 0$, the continuity equation leaves no choice of velocity, so a curve of Gaussians has exactly one flow (Exercise 4), and the rival energies of section 3 are fair comparisons. The next cell skips the shortcut and integrates $v^2\rho$ numerically, over $x$ and then $t$.

In [ ]:
x_grid = np.linspace(-12, 16, 8001)       # wide enough for the widest rival flow of section 3
t_grid = np.linspace(0, 1, 2001)


def flow_energy(m, dm, s, ds):
    # Kinetic energy of the Gaussian flow with mean m(t), std s(t) and their derivatives dm, ds.
    # Each grain keeps its standardised position, so v(t, x) = dm + (ds / s) (x - m).
    per_time = []
    for t in t_grid:
        dens = np.exp(-0.5 * ((x_grid - m(t)) / s(t)) ** 2) / (s(t) * np.sqrt(2 * np.pi))
        v = dm(t) + ds(t) / s(t) * (x_grid - m(t))
        per_time.append(np.trapezoid(v**2 * dens, x_grid))      # energy at this instant
    per_time = np.array(per_time)
    return float(np.trapezoid(per_time, t_grid)), per_time


def rival(clock=lambda t: t, dclock=lambda t: 1.0, c=0.0):
    # Gaussian path with the same two ends. `clock` changes the speed at which the straight path is run;
    # `c` adds a detour c sin(pi t) to the standard deviation (zero at both ends).
    return (
        lambda t: m0 + clock(t) * (m1 - m0),
        lambda t: dclock(t) * (m1 - m0),
        lambda t: s0 + clock(t) * (s1 - s0) + c * np.sin(np.pi * t),
        lambda t: dclock(t) * (s1 - s0) + c * np.pi * np.cos(np.pi * t),
    )


energy_geo, per_time_geo = flow_energy(*rival())
w2_squared = float(bures_wasserstein_distance([m0], [[s0**2]], [m1], [[s1**2]]) ** 2)
print(f"kinetic energy of the geodesic flow, numerically   : {energy_geo:.4f}")
print(f"squared distance W2^2 from the library            : {w2_squared:.4f}")
print(f"energy rate per instant: min {per_time_geo.min():.4f}, max {per_time_geo.max():.4f}")

**Read the output.** The numerical energy is $16.2500$, the library gives $W_2^2 = 16.25$, and the energy rate is the same at every instant. The energy rate is constant and the flow costs exactly the squared distance. Could any flow cost *less*?

## 3. Least energy: the theorem of Benamou and Brenier

Benamou and Brenier (2000) proved that the static distance has a dynamic twin. Take $\mu_0, \mu_1$ with finite second moments. Then

$$ W_2^2(\mu_0, \mu_1) = \inf_{(\rho_t, v_t)}\ \int_0^1\!\!\int |v_t(x)|^2\,\rho_t(x)\,\mathrm dx\,\mathrm dt, $$

where "inf" is the least value over all flows $(\rho_t, v_t)$ with $\rho_0 = \mu_0$, $\rho_1 = \mu_1$, obeying $\partial_t \rho_t + \nabla\!\cdot(\rho_t v_t) = 0$ ($\nabla\!\cdot$ is $\partial_x$ on the line) in the weak sense, that is tested against smooth functions, with $v_t$ square-integrable against $\rho_t$. The infimum is attained, by the McCann geodesic of `03/07`.

### Why no flow does better

**The hinge.** Over a unit time, the square of the average speed never exceeds the average of the squared speed (the Cauchy–Schwarz inequality), with equality only at constant speed. A grain covering $4$ units at constant speed $4$ pays $16$. One that stands still half the time and runs at speed $8$ the other half covers the same $4$ and pays $(0 + 64)/2 = 32$.

**One particle, then all.** Suppose the flow is regular enough for every grain to have a trajectory $X_t(x)$ with $\dot X_t = v(t, X_t)$ and $X_0(x) = x$. The continuity equation says exactly that these trajectories carry $\mu_0$ onto $\rho_t$, so, since $\rho_1 = \mu_1$, the end points carry $\mu_0$ onto $\mu_1$. Along one trajectory, the hinge gives

$$ |X_1(x) - x|^2 = \Big|\int_0^1 v\,\mathrm dt\Big|^2 \le \int_0^1 |v(t, X_t(x))|^2\,\mathrm dt , $$

with equality exactly for a straight line at constant speed. Average over $x$ with weight $\mu_0$: the right side becomes the energy of the flow, and on the left the pairs $(x, X_1(x))$ form a coupling of $\mu_0$ and $\mu_1$, which cannot cost less than the optimal one. So

$$ \mathcal E(\rho, v) \ \ge\ \int |X_1(x) - x|^2\,\mathrm d\mu_0(x)\ \ge\ W_2^2(\mu_0, \mu_1). $$

The geodesic reaches equality twice: its grains run straight at constant speed, and its coupling $(x, T(x))$ is optimal. This is the idea of the proof, not a full proof: the regularity needed to follow grains, and the passage to flows without trajectories, is left to the reference.

Test the statement on Gaussians. A rival flow with the same ends runs the straight path of mean and standard deviation with another clock, $t \mapsto t^p$, or adds to the standard deviation a detour $c\,\sin(\pi t)$, which vanishes at both ends. These rivals are the course's own illustrations; they cannot prove the theorem, since the infimum runs over all flows.

In [ ]:
exponents = np.array([1.0, 1.25, 1.5, 2.0, 2.5, 3.0])        # clock t -> t^p; p = 1 is the geodesic
clock_energy = np.array([flow_energy(*rival(clock=lambda t, p=p: t**p,
                                            dclock=lambda t, p=p: p * t ** (p - 1)))[0] for p in exponents])
detours = np.linspace(-0.6, 0.8, 8)       # c = -0.8 would push the standard deviation below zero
detour_energy = np.array([flow_energy(*rival(c=c))[0] for c in detours])
print(f"clock t^2 : energy {clock_energy[exponents == 2.0][0]:.3f}, ratio to W2^2 = {clock_energy[exponents == 2.0][0] / w2_squared:.4f}  (4/3 = {4 / 3:.4f})")
for c in (0.4, 0.8):
    print(f"detour c = {c} : energy {detour_energy[np.isclose(detours, c)][0]:.3f}")
print(f"lowest energy over all rivals tried: {min(clock_energy.min(), detour_energy.min()):.4f}  (W2^2 = {w2_squared:.4f})")
a_c2, a_c1, a_c0 = np.polyfit(detours, detour_energy, 2)
print(f"parabola fitted to the detours: energy = {a_c0:.3f} + {a_c1:.1e} c + {a_c2:.4f} c^2")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={"wspace": 0.3})
ax = axes[0]
ts = np.linspace(0, 1, 200)
ax.plot(ts, rival()[2](ts), color=viz.FLOW_COLOR, lw=2.8, label="geodesic")
ax.plot(ts, rival(clock=lambda t: t**2)[2](ts), color=viz.SOURCE_COLOR, lw=2.2, label="clock t^2")
ax.plot(ts, rival(c=0.8)[2](ts), color=viz.TARGET_COLOR, lw=2.2, label="detour c = 0.8")
ax.set_xlabel("time t")
ax.set_ylabel("standard deviation sigma(t)")
ax.set_title("Three paths with the same ends", pad=12)
ax.legend(loc="upper right")
ax = axes[1]
ax.plot(exponents, clock_energy, "o-", color=viz.SOURCE_COLOR, lw=2.2, ms=8)
ax.axhline(w2_squared, color=COLORS["muted"], ls="--", lw=1.5, label="W2^2")
ax.scatter(exponents[0], clock_energy[0], s=200, color=viz.FLOW_COLOR, zorder=4, label="geodesic clock")
ax.set_xlabel("exponent p of the clock t^p")
ax.set_ylabel("kinetic energy")
ax.set_title("Same path, other speeds", pad=12)
ax.legend(loc="upper left")
ax = axes[2]
ax.plot(detours, detour_energy, "o-", color=viz.TARGET_COLOR, lw=2.2, ms=8)
ax.axhline(w2_squared, color=COLORS["muted"], ls="--", lw=1.5, label="W2^2")
ax.scatter([0.0], [detour_energy[np.isclose(detours, 0.0)][0]], s=200, color=viz.FLOW_COLOR, zorder=4, label="geodesic")
ax.set_xlabel("detour size c")
ax.set_ylabel("kinetic energy")
ax.set_title("Same ends, a detour in width", pad=12)
ax.legend(loc="upper center")
plt.show()

**Read the figure.** On the left, the green line is the geodesic, the violet curve the same line run with the clock $t^2$, and the amber curve a path that bulges above it. In the middle, the geodesic clock $p = 1$ sits on the dashed line at $16.25$ and every other speed lies above it; the clock $t^2$ costs $21.667$, $4/3$ of $16.25$, because the hinge is strict as soon as the speed varies. On the right, the detours fall on a parabola, lowest at $c = 0$; $c = 0.4$ costs $17.040$ and $c = 0.8$ costs $19.408$. The curve is symmetric in $c$ (it stops at $-0.6$ because the standard deviation must stay positive). Exercise 3 asks you to explain the coefficient of $c^2$ that the fit returns.

## 4. Otto's reading: a geometry on distributions

Benamou and Brenier's identity has a geometric reading, due to Otto (2001). Think of distributions as points of a space and of a flow as a curve in it. The velocity field $v_t$ is the *tangent vector* of the curve at $\rho_t$. Give it the squared length

$$ \lVert v\rVert_{\rho}^2 = \int |v(x)|^2\,\rho(x)\,\mathrm dx . $$

The kinetic energy of a flow is the sum of these squared lengths along the curve, and the Benamou–Brenier formula says that $W_2$ is the length of the shortest curve, as on a surface. The McCann paths of `03/07` are the geodesics. This is a *formal* Riemannian structure: Otto treats distributions as if they formed a smooth manifold, and rigorous statements use the metric language of Ambrosio, Gigli and Savaré. On Gaussians the squared speed is $\dot m^2 + \dot\sigma^2$ (section 2), the Euclidean metric of the $(m, \sigma)$ plane: one-dimensional Gaussians form a flat piece of Wasserstein space, as `03/12` suggested (Exercise 4).

### Does the space curve?

Otto computed, formally, that the curvature is non-negative. Take a segment $\gamma$ and a point $z$. In a flat space, the squared distance from $z$ to the point at fraction $t$ of the segment is exactly the straight blend of the two end values, lowered by $t(1-t)$ times the squared length of the segment. On a positively curved space, such as the sphere of `02/08`, the points of the segment are at least as far from $z$ as that prediction: non-negative curvature means "never closer". As a formula, for every geodesic $\gamma$ and point $z$,

$$ d^2(\gamma_t, z) \ \ge\ (1 - t)\,d^2(\gamma_0, z) + t\,d^2(\gamma_1, z) - t(1 - t)\,d^2(\gamma_0, \gamma_1), $$

which is Definition 2.19 of Ambrosio and Gigli's guide. In a Euclidean space the two sides are equal. For Wasserstein space over $\mathbb R^d$ the inequality is a theorem of Ambrosio, Gigli and Savaré (2005, Thm 7.3.2); the guide's Theorem 2.20 states it for any ground space that is itself non-negatively curved. Like the Fisher–Rao geometry of `02/07`, which `02/08` turns into a piece of a sphere, Wasserstein space is non-negatively curved.

**On Gaussians.** The next cell plots, for a Gaussian geodesic $\mu_t$ and a third Gaussian $\nu$, the gap

$$ \mathrm{gap}(t) = W_2^2(\mu_t, \nu) - \Big[(1 - t)\,W_2^2(\mu_0, \nu) + t\,W_2^2(\mu_1, \nu) - t(1-t)\,W_2^2(\mu_0, \mu_1)\Big], $$

which the theorem says is never negative. Means are set to zero: a simplification, not a restriction, since the mean part of $W_2^2$ is Euclidean and the means move linearly, so it satisfies the identity exactly. Two triples are drawn: three commuting covariances, and the two matrices of `03/14` plus a third that commutes with neither. The right panel gathers the gap over 2,000 random triples, divided by $t(1-t)\,W_2^2(\mu_0, \mu_1)$ to remove the scale. The triples are the course's own illustrations; the inequality is the cited theorem.

In [ ]:
zero = np.zeros(2)


def gap(cov_0, cov_1, cov_n, t):
    # W2^2(mu_t, nu) minus the right side of the comparison inequality, all means zero.
    _, cov_t = gaussian_geodesic(zero, cov_0, zero, cov_1, t)
    return bw2(cov_t, cov_n) - ((1 - t) * bw2(cov_0, cov_n) + t * bw2(cov_1, cov_n) - t * (1 - t) * bw2(cov_0, cov_1))


def bw2(a, b):
    return bures_wasserstein_distance(zero, a, zero, b) ** 2


def comm(a, b):
    return np.linalg.norm(a @ b - b @ a)
C0 = np.array([[1.5, 0.4], [0.4, 0.6]])         # the two covariances of 03/14
C1 = np.array([[0.5, -0.3], [-0.3, 1.4]])
Cn = np.array([[1.0, 0.6], [0.6, 1.2]])         # a third one, which commutes with neither
D0, D1, Dn = np.diag([1.0, 4.0]), np.diag([9.0, 1.0]), np.diag([2.0, 3.0])   # three commuting ones

ts = np.linspace(0, 1, 101)
gap_noncommuting = np.array([gap(C0, C1, Cn, t) for t in ts])
gap_diagonal = np.array([gap(D0, D1, Dn, t) for t in ts])
print(f"largest commutator norm among C0, C1, Cn: {max(comm(C0, C1), comm(C0, Cn), comm(C1, Cn)):.3f};   among D0, D1, Dn: {max(comm(D0, D1), comm(D0, Dn), comm(D1, Dn)):.1f}")
for t in (0.25, 0.5, 0.75):
    print(f"t = {t:.2f}   gap, three non-commuting covariances: {gap(C0, C1, Cn, t):.4f}   commuting (diagonal): {gap(D0, D1, Dn, t):.1e}")
print(f"commuting triple, largest |gap| over 101 times: {np.abs(gap_diagonal).max():.1e}")
mid = gap(C0, C1, Cn, 0.5)
print(f"scale of the left curve: midpoint gap {mid:.4f} against the flat-space term t(1-t) W2^2(mu_0, mu_1) = {0.25 * bw2(C0, C1):.4f}, ratio {mid / (0.25 * bw2(C0, C1)):.3f}")

# Two checks on the scope of "zero": all three covariances must commute, not only the two ends.
theta = 0.6
Q = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
R0, R1, Rn = (Q @ D @ Q.T for D in (D0, D1, Dn))                  # the same commuting triple, in a tilted common basis
print("commuting triple in a tilted basis (not diagonal): gaps at t = 0.25, 0.5, 0.75 = "
      + ", ".join(f"{gap(R0, R1, Rn, t):.1e}" for t in (0.25, 0.5, 0.75)))
Cn_tilted = np.array([[2.0, 0.9], [0.9, 3.0]])                    # does not commute with the diagonal ends
print(f"diagonal ends D0, D1 with a third that does not commute (commutator {comm(D0, Cn_tilted):.2f}): gap at t = 0.5 = {gap(D0, D1, Cn_tilted, 0.5):.4f}")

normalised, times = [], []
for _ in range(2000):                           # random positive-definite covariances, G G^T + 0.05 I
    covs = [(lambda G: G @ G.T + 0.05 * np.eye(2))(rng.normal(size=(2, 2))) for _ in range(3)]
    t = rng.uniform(0.05, 0.95)
    normalised.append(gap(*covs, t) / (t * (1 - t) * bw2(covs[0], covs[1])))
    times.append(t)
normalised = np.array(normalised)
print(f"2,000 random triples: smallest normalised gap {normalised.min():.1e}, largest {normalised.max():.2f}, negative: {int((normalised < 0).sum())}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"wspace": 0.25})
ax = axes[0]
ax.plot(ts, gap_noncommuting, color=viz.FLOW_COLOR, lw=2.8, label="covariances that do not commute")
ax.plot(ts, gap_diagonal, color=viz.SOURCE_COLOR, lw=2.8, ls="--", label="three commuting covariances")
ax.axhline(0, color=COLORS["muted"], lw=1.2)
ax.set_xlabel("time t along the geodesic")
ax.set_ylabel("gap of the comparison inequality")
ax.set_title("Zero when all three covariances commute", pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=1)
ax = axes[1]
ax.hist(normalised, bins=40, color=viz.TARGET_COLOR, edgecolor="white")
ax.axvline(0, color=COLORS["text"], lw=1.6)
ax.set_xlabel("gap / [ t(1-t) W2^2(mu_0, mu_1) ]")
ax.set_ylabel("number of random triples")
ax.set_title("Normalised gap for 2,000 random triples", pad=12)
plt.show()

**Read the figure.** On the left, the dashed violet line lies on zero, to rounding error ($10^{-15}$ in the printout). Three covariances that share an eigenbasis give $W_2^2$ as a squared Euclidean distance between vectors of standard deviations, the geodesic is a straight segment, and the inequality is the Euclidean identity. The printout shows the same zero in a tilted shared basis, and a positive gap when the third covariance does not commute with the diagonal ends: what matters is that all three commute. The green curve is the non-commuting triple. It vanishes at both ends, where $\mu_t$ is an end point, and rises to $0.0109$ at the midpoint, against a flat-space term of $0.1794$. On the right, the normalised gap of 2,000 random triples lies entirely to the right of the line at zero: the smallest value is positive and none is negative. The figure tests one inequality on Gaussians; it does not measure the curvature of the whole space.

## 5. The quantum counterpart, named and not built

As in `03/13`, the flow picture has a quantum counterpart. On density matrices, Carlen and Maas (2014) built a quantum continuity equation for paths of states and, on those paths, a quantum Riemannian metric that plays the part of Otto's. This course names that construction and does not build it. `04/03` places it on the map of quantum transport, and the course builds a different object, the coupling, from `04/04` onward.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| Benamou–Brenier / Otto Riemannian metric | Carlen–Maas quantum Riemannian metric (not built in this course) |

## Your turn

1. **A pure dilation.** Take the flow from $\mathcal N(0, 1)$ to $\mathcal N(0, 4^2)$, with map $T(x) = 4x$. Check with pen and paper that its velocity field is $3x/(1 + 3t)$ and obeys the continuity equation, and show that the energy is $(\sigma_1 - \sigma_0)^2 = 9$.
2. **The price of a clock.** Run the straight path of section 1 with the clock $t \mapsto t^p$. Show that the energy is $p^2/(2p - 1)$ times $W_2^2$ for $p > \tfrac12$ (hint: $\int_0^1 p^2 t^{2p-2}\,\mathrm dt$). For which $p$ is it smallest? Check $21.667$ at $p = 2$.
3. **The price of a detour.** Add the detour $c\,\sin(\pi t)$ to the standard deviation, keeping the means on their straight line. Use $\int|v|^2\rho\,\mathrm dx = \dot m^2 + \dot\sigma^2$ to compute the energy by hand, explain why the term linear in $c$ vanishes, and compare your coefficient of $c^2$ with the fitted one.
4. **Gaussians on a line form a flat space (harder).** Show that $(m, \sigma) \mapsto \mathcal N(m, \sigma^2)$ turns the Wasserstein geometry of one-dimensional Gaussians into the Euclidean geometry of the half-plane $\sigma > 0$. Steps: (a) in one dimension the continuity equation fixes the velocity: the flux $\rho v$ must tend to $0$ at $-\infty$, so $\rho v(t, x) = -\int_{-\infty}^{x}\partial_t\rho\,\mathrm dy$; write $\rho = \varphi((x - m)/\sigma)/\sigma$, differentiate in $t$, integrate, and recover $v = \dot m + (\dot\sigma/\sigma)(x - m)$; (b) deduce that the squared speed of the curve is $\dot m^2 + \dot\sigma^2$; (c) conclude with the Benamou–Brenier formula and the distance of `03/12`. Open stretch: in two dimensions a field $w$ with $\nabla\!\cdot(\rho w) = 0$ can be added to $v$ without breaking the equation. What does that change, given the positive gap of section 4?

## What you built

- You can write the continuity equation from a count of mass and check it on the Gaussian geodesic flow.
- You can compute a flow's kinetic energy and show that the geodesic flow costs exactly $W_2^2$.
- You can state the theorem of Benamou and Brenier, give the particle argument behind it, and test it on rival flows.
- You can read Otto's picture and see the non-negative curvature inequality hold on Gaussians.
- You can say where Carlen and Maas stand: named, not built.

**Module 3 is complete.** You started from Monge's deterministic map and the Kantorovich LP, with Birkhoff's theorem behind it. You then built the Wasserstein metric and its geodesics, duality, the Sinkhorn algorithm and Amari's bridge. You closed with the Gaussian closed form, the Bures bridge and the flow picture of this notebook. That is the core of classical optimal transport, together with the transport rows of the dictionary that will let it speak quantum. It is a substantial body of understanding, and it is the launchpad for everything ahead.

## What's next

Module 4 — **quantum optimal transport** — carries the coupling, the Kantorovich program, the Wasserstein distance and the Sinkhorn iteration to density matrices. It opens by asking *why* a genuinely quantum transport is needed at all: what the naive "diagonal" approach misses, and how the $|+\rangle$-vs-mixed lesson and the Bures bridge you built point the way to quantum Wasserstein. `04/03` then places Carlen and Maas on its map of quantum transport.

## References

- J.-D. Benamou & Y. Brenier, "A computational fluid mechanics solution to the Monge–Kantorovich mass transfer problem", *Numer. Math.* 84, 375–393 (2000). DOI:10.1007/s002110050002
- F. Otto, "The geometry of dissipative evolution equations: the porous medium equation", *Comm. PDE* 26, 101–174 (2001). DOI:10.1081/PDE-100002243
- L. Ambrosio & N. Gigli, "A user's guide to optimal transport", in *Modelling and Optimisation of Flows on Networks*, Lecture Notes in Mathematics, Springer (2013), 1–155. DOI:10.1007/978-3-642-32160-3_1. Numbering of the SISSA copy `users_guide-final.pdf`; the Springer chapter may differ. Proposition 2.30 states the Benamou–Brenier formula, and Definition 2.19 and Theorem 2.20 are the curvature statements of section 4.
- E. A. Carlen & J. Maas, "An analog of the 2-Wasserstein metric in non-commutative probability under which the fermionic Fokker–Planck equation is gradient flow for the entropy", *Comm. Math. Phys.* 331, 887–926 (2014). DOI:10.1007/s00220-014-2124-8
- L. Ambrosio, N. Gigli & G. Savaré, *Gradient Flows in Metric Spaces and in the Space of Probability Measures*, Lectures in Mathematics ETH Zürich, Birkhäuser (2005), §7.3. DOI:10.1007/b137080
- R. McCann, "A convexity principle for interacting gases", *Adv. Math.* 128, 153–179 (1997). DOI:10.1006/aima.1997.1634

**Previous:** `notebooks/03_ClassicalOptimalTransport/14_gaussian_geodesic.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/01_diagonal_collapse.ipynb`